# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

## 2. Pipeline de procesamiento

In [ ]:
tree_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
scaled_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Debe estar dentro del pipeline para que se ajuste únicamente con los datos de entrenamiento de cada fold. Si se escala antes de la validación cruzada, la media y desviación estándar de los folds de validación contaminan el entrenamiento y producen una estimación optimista.

## 3. Modelos candidatos

In [ ]:
models = {
    "decision_tree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", KNeighborsClassifier()),
    ]),
}


## 4. Comparación inicial con validación cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_rows = []
for name, estimator in models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True
    )
    initial_rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "validation_accuracy_mean": scores["test_score"].mean(),
        "validation_accuracy_std": scores["test_score"].std(),
    })
initial_comparison = pd.DataFrame(initial_rows).sort_values("validation_accuracy_mean", ascending=False)
print(initial_comparison)


## 5. Optimización de hiperparámetros

In [ ]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": [1, 2, 5],
    },
    "logistic_regression": {
        "model__C": [0.1, 1.0, 10.0],
    },
    "knn": {
        "model__n_neighbors": [3, 5, 9, 15],
        "model__weights": ["uniform", "distance"],
    },
}
searches = {}
tuned_rows = []
for name, estimator in models.items():
    search = GridSearchCV(estimator, param_grids[name], cv=cv, scoring="accuracy", n_jobs=1)
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_rows.append({
        "model": name,
        "best_score": search.best_score_,
        "best_params": json.dumps(search.best_params_),
    })
tuned_comparison = pd.DataFrame(tuned_rows).sort_values("best_score", ascending=False)
print(tuned_comparison)


## 6. Selección y persistencia del campeón

In [ ]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_
champion_score = float(searches[champion_name].best_score_)
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")
training_metadata = {
    "champion": champion_name,
    "cv_accuracy": champion_score,
    "best_params": searches[champion_name].best_params_,
    "random_state": RANDOM_STATE,
    "selection_rule": "highest mean validation accuracy in 5-fold stratified CV",
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(training_metadata, indent=2, default=str), encoding="utf-8")
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Campeón:", champion_name, "CV accuracy:", champion_score)


In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

> Se selecciona el modelo con mayor accuracy media en validación cruzada. También se revisan la desviación entre folds y la diferencia entre accuracy de entrenamiento y validación para detectar sobreajuste. El pipeline conserva todo el preprocesamiento y ofrece una solución reproducible; si el desempeño es similar, se prefiere el modelo más simple e interpretable.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.